In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
import warnings
import os
import gc
import shutil
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
label2id = {
    "Clear Reply": 0,
    "Ambivalent": 1,
    "Clear Non-Reply": 2
}
id2label = {v: k for k, v in label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1': f1}

In [ ]:
class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
original_ds = load_dataset("ailsntua/QEvasion")

original_train_df = original_ds['train'].to_pandas()
original_val_df = original_ds['validation'].to_pandas()
original_test_df = original_ds['test'].to_pandas()

train_df_orig, val_df_orig = train_test_split(
    original_train_df,
    test_size=700,
    random_state=SEED,
    stratify=original_train_df['clarity_label']
)

In [ ]:
from huggingface_hub import login

login()

In [ ]:
augmented_gpt_ds = load_dataset("gabrielstefan04/qevasion-gpt5.1-augmented")
aug_gpt_train_df = augmented_gpt_ds['train'].to_pandas()
aug_gpt_val_df = augmented_gpt_ds['validation'].to_pandas()

In [ ]:
augmented_bt_ds = load_dataset("gabrielstefan04/qevasion-backtranslation-augmented")
aug_bt_train_df = augmented_bt_ds['train'].to_pandas()
aug_bt_val_df = augmented_bt_ds['validation'].to_pandas()

In [ ]:
DEBERTA_PARAMS = {
    "learning_rate": 1.9906996673933376e-06,
    "weight_decay": 0.09507143064099162,
    "warmup_ratio": 0.15979909127171077,
    "gradient_accumulation_steps": 1,
}

In [ ]:
def prepare_datasets(train_df, val_df, tokenizer, max_length):
    
    def tokenize_function(examples):
        inputs = [
            f"Question: {q}\nAnswer: {a}"
            for q, a in zip(examples["question"], examples["interview_answer"])
        ]
        tokenized = tokenizer(
            inputs,
            truncation=True,
            max_length=max_length,
            padding=False
        )
        tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
        return tokenized
    
    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
    
    train_tokenized = train_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=train_dataset.column_names
    )
    val_tokenized = val_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=val_dataset.column_names
    )
    
    return train_tokenized, val_tokenized


def train_model(model_name, train_tokenized, val_tokenized, tokenizer, params, output_dir, max_length):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
    )
    model.config.use_cache = False
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        
        learning_rate=params["learning_rate"],
        weight_decay=params["weight_decay"],
        warmup_ratio=params["warmup_ratio"],
        gradient_accumulation_steps=params["gradient_accumulation_steps"],
        
        num_train_epochs=15,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        
        logging_steps=50,
        logging_strategy="steps",
        report_to="none",
        
        fp16=False,
        bf16=True,
        optim="adamw_torch",
        seed=SEED,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1", greater_is_better=True)],
    )
    
    trainer.train()
    
    return trainer, model


def evaluate_on_test(trainer, tokenizer, test_df, max_length, model_name, dataset_name):
    def tokenize_test(examples):
        texts = [
            f"Question: {q}\nAnswer: {a}"
            for q, a in zip(examples['question'], examples['interview_answer'])
        ]
        return tokenizer(texts, truncation=True, padding=False, max_length=max_length)
    
    test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
    test_tokenized = test_dataset.map(
        tokenize_test,
        batched=True,
        remove_columns=[col for col in test_dataset.column_names if col not in ['clarity_label']]
    )
    
    predictions_output = trainer.predict(test_tokenized)
    predictions = np.argmax(predictions_output.predictions, axis=-1)
    predicted_labels = [id2label[pred] for pred in predictions]
    
    y_true = test_df['clarity_label'].tolist()
    y_pred = predicted_labels
    
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=clarity_labels, zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=clarity_labels, zero_division=0)
    
    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=clarity_labels, zero_division=0))
    
    print("Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=clarity_labels)
    print(f"Labels: {clarity_labels}")
    print(cm)
    
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    }


### 1. DeBERTa-v3-large on Original Dataset

In [ ]:
DEBERTA_MODEL = "microsoft/deberta-v3-large"
DEBERTA_MAX_LENGTH = 512

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)

In [ ]:
train_orig_db, val_orig_db = prepare_datasets(
    train_df_orig, val_df_orig, 
    deberta_tokenizer, 
    DEBERTA_MAX_LENGTH
)

In [ ]:
trainer_db_orig, model_db_orig = train_model(
    DEBERTA_MODEL,
    train_orig_db,
    val_orig_db,
    deberta_tokenizer,
    DEBERTA_PARAMS,
    output_dir="./results_deberta_original",
    max_length=DEBERTA_MAX_LENGTH
)

In [ ]:
results_db_orig = evaluate_on_test(
    trainer_db_orig,
    deberta_tokenizer,
    original_test_df,
    DEBERTA_MAX_LENGTH,
    "DeBERTa-v3-large",
    "Original Dataset → Test"
)

In [ ]:
del model_db_orig, trainer_db_orig
torch.cuda.empty_cache()
gc.collect()

if os.path.exists("./results_deberta_original"):
    shutil.rmtree("./results_deberta_original")

### 2. DeBERTa-v3-large on GPT-5.1 Augmented Dataset

In [ ]:
train_aug_gpt_db, val_aug_gpt_db = prepare_datasets(
    aug_gpt_train_df, aug_gpt_val_df,
    deberta_tokenizer,
    DEBERTA_MAX_LENGTH
)

In [ ]:
trainer_db_gpt, model_db_gpt = train_model(
    DEBERTA_MODEL,
    train_aug_gpt_db,
    val_aug_gpt_db,
    deberta_tokenizer,
    DEBERTA_PARAMS,
    output_dir="./results_deberta_gpt_augmented",
    max_length=DEBERTA_MAX_LENGTH
)

In [ ]:
results_db_gpt = evaluate_on_test(
    trainer_db_gpt,
    deberta_tokenizer,
    original_test_df,  
    DEBERTA_MAX_LENGTH,
    "DeBERTa-v3-large",
    "GPT-5.1 Augmented → Test"
)

In [ ]:
del model_db_gpt, trainer_db_gpt
torch.cuda.empty_cache()
gc.collect()

if os.path.exists("./results_deberta_gpt_augmented"):
    shutil.rmtree("./results_deberta_gpt_augmented")

### 3. DeBERTa-v3-large on Back-translation Augmented Dataset

In [ ]:
train_aug_bt_db, val_aug_bt_db = prepare_datasets(
    aug_bt_train_df, aug_bt_val_df,
    deberta_tokenizer,
    DEBERTA_MAX_LENGTH
)

In [ ]:

trainer_db_bt, model_db_bt = train_model(
    DEBERTA_MODEL,
    train_aug_bt_db,
    val_aug_bt_db,
    deberta_tokenizer,
    DEBERTA_PARAMS,
    output_dir="./results_deberta_backtranslation_augmented",
    max_length=DEBERTA_MAX_LENGTH
)

In [ ]:
results_db_bt = evaluate_on_test(
    trainer_db_bt,
    deberta_tokenizer,
    original_test_df,  
    DEBERTA_MAX_LENGTH,
    "DeBERTa-v3-large",
    "Back-translation Augmented → Test"
)

In [ ]:
del model_db_bt, trainer_db_bt
torch.cuda.empty_cache()
gc.collect()

if os.path.exists("./results_deberta_backtranslation_augmented"):
    shutil.rmtree("./results_deberta_backtranslation_augmented")


### Results Summary

In [ ]:
all_results = [
    {"Training Data": "Original", **results_db_orig},
    {"Training Data": "GPT-5.1 Augmented", **results_db_gpt},
    {"Training Data": "Back-translation Augmented", **results_db_bt}
]

results_df = pd.DataFrame(all_results)

print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))